<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-12-production-deploy/lesson-12.2-rag-backend/practice/GCP_Capstone_12.2_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 12.2 — RAG API Backend

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup: install, authenticate, configure

Run this **once** at the top. It installs the pinned dependencies from the lesson's `requirements.txt`, authenticates with **Application Default Credentials** (no API keys), and sets the environment variables that `config.py` reads. Replace `documind-ai-YOUR-ID` and the Vector Search resource IDs with your own before running the live retrieval/generation cells.

In [ ]:
%%bash
pip install -q \
  google-genai==2.20.0 \
  google-cloud-aiplatform==1.95.0 \
  google-cloud-firestore==2.20.0 \
  'google-cloud-discoveryengine>=0.13.0' \
  fastapi==0.115.6 'uvicorn[standard]==0.32.1' \
  pydantic==2.10.4 pydantic-settings==2.7.0 \
  opentelemetry-exporter-gcp-trace==1.9.0 \
  opentelemetry-instrumentation-fastapi==0.51b0 \
  httpx==0.28.1
echo 'deps installed'

In [ ]:
import os

# --- Application Default Credentials (Colab only) ---
try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via ADC')
except ImportError:
    print('Not on Colab — assuming ADC is already configured (gcloud auth application-default login)')

# --- Environment for config.py (edit these) ---
os.environ['GOOGLE_CLOUD_PROJECT']        = 'documind-ai-YOUR-ID'
os.environ['VECTOR_INDEX_ENDPOINT']       = 'projects/documind-ai-YOUR-ID/locations/us-central1/indexEndpoints/0000000000000000000'
os.environ['VECTOR_DEPLOYED_INDEX_ID']    = 'dep_001'

USD_INR = 85  # for any cost display in INR
print('PROJECT =', os.environ['GOOGLE_CLOUD_PROJECT'])

### `config.py` — shared Pydantic settings

Every module below imports `settings` from here (embed model, generator model, rerank model, top-k bounds). Lifted from **Cell 2** of the lesson notebook. Written to disk so `retriever.py`, `generator.py`, and `main.py` can `from config import settings`.

In [ ]:
%%writefile config.py
from pydantic import Field
from pydantic_settings import BaseSettings, SettingsConfigDict

class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env", extra="ignore")

    project_id: str = Field(alias="GOOGLE_CLOUD_PROJECT")
    region: str = "us-central1"
    india_region: str = "asia-south1"
    vector_index_endpoint: str = Field(alias="VECTOR_INDEX_ENDPOINT")
    vector_deployed_index: str = Field(alias="VECTOR_DEPLOYED_INDEX_ID")
    embed_model: str = "text-embedding-005"
    generator_model: str = "gemini-3.6-flash"
    rerank_model: str = "semantic-ranker-fast-004"
    top_k_retrieve: int = 20
    top_k_rerank: int = 5
    max_context_tokens: int = 8000
    max_answer_tokens: int = 1024

settings = Settings()

## Exercise 1: FastAPI skeleton + /health

**Difficulty:** Easy

Write `main.py` with a FastAPI app, `GET /health`, and `POST /v1/query` that echoes the request (no retrieval yet). Run locally with `uvicorn main:app --reload`.

**Steps:**
1. Create a FastAPI app.
2. Add `GET /health` returning `{"status":"ok"}`.
3. Add `POST /v1/query` that accepts the request body and echoes it back (retrieval comes in later exercises).
4. Exercise it. (Locally you'd `uvicorn main:app --reload`; in Colab we drive it with `TestClient` so the cell is runnable.)

**Expected:** `GET /health` returns `{"status":"ok"}`. `/v1/query` accepts a `QueryRequest` and echoes it.

In [ ]:
# Skeleton version of main.py (Cell 6 grows this into the full app in Ex 5-7).
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

app = FastAPI(title="DocuMind API", version="0.1.0")

class EchoRequest(BaseModel):
    query: str = Field(min_length=1)
    tenant_id: str

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/v1/query")
def query(req: EchoRequest):
    return {"echo": req.model_dump()}

# Drive it in-process (equivalent to `curl` against `uvicorn main:app`)
c = TestClient(app)
print('GET  /health   ->', c.get('/health').json())
print('POST /v1/query ->', c.post('/v1/query', json={'query': 'hello', 'tenant_id': 'tenant-acme'}).json())

## Exercise 2: Pydantic request/response schemas

**Difficulty:** Easy

Define `QueryRequest` (min/max lengths, top_k bounds), `Citation` (score 0-1), `RAGAnswer` (enum confidence). Confirm FastAPI's auto-generated `/docs` renders them. Send a malformed request — verify a 422 response.

**Steps:**
1. Write `schemas.py` with `QueryRequest`, `Citation`, `RAGAnswer`, `StreamEvent`.
2. Constrain fields: `query` length 1-4000, `top_k` between 1 and 20, `score` in [0, 1], `confidence` an enum.
3. Mount the schemas on a FastAPI route so `/docs` renders the typed models.
4. Send `query=""` and `top_k=100` — confirm each returns 422.

**Expected:** `/docs` shows typed schemas. `query=""` returns 422 ("String should have at least 1 character"). `top_k=100` returns 422 ("less than or equal to 20").

In [ ]:
%%writefile schemas.py
from typing import Literal, Optional, List
from pydantic import BaseModel, Field

class QueryRequest(BaseModel):
    query: str = Field(min_length=1, max_length=4000)
    tenant_id: str = Field(min_length=1)
    user_id: str = Field(min_length=1)
    top_k: int = Field(default=5, ge=1, le=20)
    stream: bool = True
    filters: Optional[dict] = None  # e.g. {"doc_type": "policy"}

class Citation(BaseModel):
    chunk_id: str
    source_uri: str
    page: Optional[int] = None
    quote: str = Field(max_length=500)
    score: float = Field(ge=0, le=1)

class RAGAnswer(BaseModel):
    answer: str
    citations: List[Citation]
    confidence: Literal["high", "medium", "low"]
    answerable: bool
    model: str
    tokens_in: int
    tokens_out: int
    latency_ms: int

class StreamEvent(BaseModel):
    # Server-Sent Events payload
    event: Literal["token", "citation", "done", "error"]
    data: dict

In [ ]:
# Verify the constraints reject malformed input with a 422.
from fastapi import FastAPI
from fastapi.testclient import TestClient
from schemas import QueryRequest

docs_app = FastAPI()
@docs_app.post('/v1/query')
def q(req: QueryRequest):
    return {'ok': True}

c = TestClient(docs_app)

bad_empty = c.post('/v1/query', json={'query': '', 'tenant_id': 't', 'user_id': 'u'})
bad_topk  = c.post('/v1/query', json={'query': 'hi', 'tenant_id': 't', 'user_id': 'u', 'top_k': 100})
good      = c.post('/v1/query', json={'query': 'hi', 'tenant_id': 't', 'user_id': 'u', 'top_k': 5})

print('empty query :', bad_empty.status_code, bad_empty.json()['detail'][0]['msg'])
print('top_k=100   :', bad_topk.status_code, bad_topk.json()['detail'][0]['msg'])
print('valid       :', good.status_code, good.json())
print('\nOpenAPI schema keys:', list(docs_app.openapi()['components']['schemas'].keys()))

## Exercise 3: Query embedding + ANN retrieve

**Difficulty:** Medium

Implement `embed_query` with `task_type=RETRIEVAL_QUERY`. Implement `retrieve` with a `tenant_id` restrict on the Matching Engine index. Fan out to Firestore for chunk payloads. Measure p50 latency over 50 queries.

**Steps:**
1. Build a lazily-cached `google-genai` Vertex client (`enterprise=True`) and index-endpoint / Firestore handles.
2. `embed_query`: call `client.models.embed_content` with `task_type="RETRIEVAL_QUERY"`, 768 dims.
3. `retrieve`: embed, ANN-search the deployed index with a `tenant_id` restrict, fan out to Firestore `chunks`.
4. Time 50 `retrieve` calls and report the p50.

**Expected:** Embedding ~40ms, ANN ~30ms, Firestore fan-out ~25ms; total retrieve p50 < 120ms. A tenant-B restrict returns zero rows against tenant-A data.

In [ ]:
%%writefile retriever.py
# retriever.py — embed + ANN retrieve. Ex 4 rewrites this file to add rerank().
from functools import lru_cache
from google.cloud import aiplatform
from google import genai
from google.genai import types
from google.cloud import firestore
from config import settings

@lru_cache(maxsize=1)
def _genai_client():
    return genai.Client(enterprise=True, project=settings.project_id, location=settings.region)

@lru_cache(maxsize=1)
def _index_endpoint():
    return aiplatform.MatchingEngineIndexEndpoint(settings.vector_index_endpoint)

@lru_cache(maxsize=1)
def _fs():
    return firestore.Client(project=settings.project_id, database="(default)")

def embed_query(q: str) -> list[float]:
    resp = _genai_client().models.embed_content(
        model=settings.embed_model, contents=q,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=768))
    return resp.embeddings[0].values

def retrieve(query: str, tenant_id: str, top_k: int, filters: dict | None = None) -> list[dict]:
    vec = embed_query(query)
    restricts = [{"namespace": "tenant_id", "allow": [tenant_id]}]
    if filters:
        for k, v in filters.items():
            restricts.append({"namespace": k, "allow": [str(v)]})
    resp = _index_endpoint().find_neighbors(
        deployed_index_id=settings.vector_deployed_index,
        queries=[vec], num_neighbors=settings.top_k_retrieve,
        filter=restricts,
    )
    ids = [n.id for n in resp[0]]
    scores = {n.id: n.distance for n in resp[0]}
    # Fan-out to Firestore for chunk payloads
    chunks = []
    for doc in _fs().collection("chunks").where("__name__", "in", ids[:10]).stream():
        d = doc.to_dict(); d["id"] = doc.id; d["score"] = scores.get(doc.id, 0)
        chunks.append(d)
    return chunks

In [ ]:
# Latency harness. Needs a live deployed Vector Search index + Firestore `chunks`
# for your tenant; guarded so the notebook still runs without that infra.
import time, statistics
try:
    import importlib, retriever; importlib.reload(retriever)
    lat = []
    for _ in range(50):
        t0 = time.time()
        retriever.retrieve('Which file types are supported?', 'tenant-acme', 5)
        lat.append((time.time() - t0) * 1000)
    print(f'retrieve p50: {statistics.median(lat):.0f} ms  (target < 120ms)')
    # tenant isolation spot-check
    empty = retriever.retrieve('secret', 'tenant-does-not-exist', 5)
    print('cross-tenant rows (expect 0):', len(empty))
except Exception as e:
    print('Live retrieve needs a deployed index + Firestore. Module is ready. Error:', type(e).__name__, e)

## Exercise 4: Rerank with semantic-ranker-fast-004

**Difficulty:** Medium

Pass the top-20 ANN results to the Vertex AI Ranking API (`semantic-ranker-fast-004`), keep the top 5, and log the before/after order. On a curated test set, measure how many "correct" chunks rerank lifts into the top-5.

**Steps:**
1. Add `rerank(query, chunks, k)` to `retriever.py` using `discoveryengine.RankServiceClient`.
2. Build `RankingRecord`s from the ANN chunks, call `.rank(model="semantic-ranker-fast-004", top_n=k, ...)`.
3. Attach `rerank_score` and return the reordered chunks.
4. Log the order change; measure precision@5 uplift on a curated set.

**Expected:** Rerank p50 ~110ms. On the curated set precision@5 rises from ~0.62 (raw ANN) to ~0.84 (ANN + rerank); ordering changes for ~7/20 queries.

In [ ]:
%%writefile retriever.py
# Full retriever.py: helpers + embed_query + retrieve + rerank (completes Ex 3's module).
from functools import lru_cache
from google.cloud import aiplatform
from google.cloud import discoveryengine_v1 as discoveryengine
from google import genai
from google.genai import types
from google.cloud import firestore
from config import settings

@lru_cache(maxsize=1)
def _genai_client():
    return genai.Client(enterprise=True, project=settings.project_id, location=settings.region)

@lru_cache(maxsize=1)
def _index_endpoint():
    return aiplatform.MatchingEngineIndexEndpoint(settings.vector_index_endpoint)

@lru_cache(maxsize=1)
def _fs():
    return firestore.Client(project=settings.project_id, database="(default)")

def embed_query(q: str) -> list[float]:
    resp = _genai_client().models.embed_content(
        model=settings.embed_model, contents=q,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=768))
    return resp.embeddings[0].values

def retrieve(query: str, tenant_id: str, top_k: int, filters: dict | None = None) -> list[dict]:
    vec = embed_query(query)
    restricts = [{"namespace": "tenant_id", "allow": [tenant_id]}]
    if filters:
        for k, v in filters.items():
            restricts.append({"namespace": k, "allow": [str(v)]})
    resp = _index_endpoint().find_neighbors(
        deployed_index_id=settings.vector_deployed_index,
        queries=[vec], num_neighbors=settings.top_k_retrieve,
        filter=restricts,
    )
    ids = [n.id for n in resp[0]]
    scores = {n.id: n.distance for n in resp[0]}
    chunks = []
    for doc in _fs().collection("chunks").where("__name__", "in", ids[:10]).stream():
        d = doc.to_dict(); d["id"] = doc.id; d["score"] = scores.get(doc.id, 0)
        chunks.append(d)
    return chunks

@lru_cache(maxsize=1)
def _ranker():
    return discoveryengine.RankServiceClient()

def rerank(query: str, chunks: list[dict], k: int) -> list[dict]:
    if not chunks: return chunks
    client = _ranker()
    ranking_config = client.ranking_config_path(
        project=settings.project_id, location="global",
        ranking_config="default_ranking_config")
    records = [discoveryengine.RankingRecord(id=str(i), content=c["text"])
               for i, c in enumerate(chunks)]
    resp = client.rank(request=discoveryengine.RankRequest(
        ranking_config=ranking_config,
        model=settings.rerank_model,
        top_n=k, query=query, records=records))
    out = []
    for r in resp.records:
        chunks[int(r.id)]["rerank_score"] = r.score
        out.append(chunks[int(r.id)])
    return out

In [ ]:
# Rerank demo: feed a few candidate chunks, log the reorder. Needs the Ranking API
# enabled on your project; guarded so the notebook still runs otherwise.
sample_chunks = [
    {'id': 'c1', 'text': 'DocuMind accepts PDF, DOCX, TXT and Markdown uploads.', 'score': 0.55},
    {'id': 'c2', 'text': 'Billing is prorated monthly per active seat.',            'score': 0.61},
    {'id': 'c3', 'text': 'Supported file types include PDF and scanned images (OCR).','score': 0.48},
]
try:
    import importlib, retriever; importlib.reload(retriever)
    print('before:', [c['id'] for c in sample_chunks])
    ranked = retriever.rerank('Which file types are supported?', sample_chunks, k=5)
    print('after :', [(c['id'], round(c['rerank_score'], 3)) for c in ranked])
except Exception as e:
    print('Live rerank needs the Vertex AI Ranking API. Module is ready. Error:', type(e).__name__, e)

## Exercise 5: Structured generation with RAGAnswer schema

**Difficulty:** Medium

Implement `generate`. Build the context with numbered headers like `[1] source page 3`. Call Gemini 3.6 Flash with a `response_schema` that includes `answerable: bool`. Test with an out-of-scope question and confirm `answerable=false`.

**Steps:**
1. Write `generator.py` with a strict system prompt (answer only from numbered context, cite `[N]`, decline when unsupported).
2. `build_context`: number each chunk `[i] source_uri page N`.
3. `generate`: call `client.models.generate_content` with JSON `response_schema` (answer / citations / confidence / answerable) and `thinking_budget=0`.
4. Map integer citations back to `Citation` objects; test an out-of-scope query.

**Expected:** In-scope query -> `answerable=true`, 2-3 citations, `confidence=high`. Out-of-scope -> `answerable=false`, `citations=[]`, `confidence=low`, answer politely declines.

In [ ]:
%%writefile generator.py
from google import genai
from google.genai import types
from schemas import RAGAnswer, Citation
from config import settings

_client = genai.Client(enterprise=True, project=settings.project_id, location=settings.region)

SYSTEM = """You are DocuMind, a retrieval-grounded assistant.
Rules:
1. Answer ONLY from the numbered context below. Never invent sources.
2. Cite using [N] where N is the chunk number. Multiple chunks: [1,2].
3. If the context does not contain the answer, set answerable=false and say so.
4. Keep answers under 300 words unless asked for more.
"""

def build_context(chunks: list[dict]) -> str:
    lines = []
    for i, c in enumerate(chunks, 1):
        src = c.get("source_uri", "")
        page = c.get("page_start")
        header = f"[{i}] {src}" + (f" page {page}" if page else "")
        lines.append(f"{header}\n{c['text']}")
    return "\n\n".join(lines)

def generate(query: str, chunks: list[dict]) -> RAGAnswer:
    context = build_context(chunks)
    prompt = f"{SYSTEM}\n\nContext:\n{context}\n\nQuestion: {query}"

    r = _client.models.generate_content(
        model=settings.generator_model,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema={
                "type": "OBJECT",
                "properties": {
                    "answer":     {"type": "STRING"},
                    "citations":  {"type": "ARRAY", "items": {"type": "INTEGER"}},
                    "confidence": {"type": "STRING", "enum": ["high","medium","low"]},
                    "answerable": {"type": "BOOLEAN"},
                },
                "required": ["answer","citations","confidence","answerable"],
            },
            temperature=0.1,
            max_output_tokens=settings.max_answer_tokens,
            thinking_config=types.ThinkingConfig(thinking_budget=0),
        ),
    )
    parsed = r.parsed or {}
    cits = []
    for idx in parsed.get("citations", []):
        if 1 <= idx <= len(chunks):
            c = chunks[idx - 1]
            cits.append(Citation(chunk_id=c["id"], source_uri=c["source_uri"],
                                 page=c.get("page_start"), quote=c["text"][:500],
                                 score=float(c.get("rerank_score") or c.get("score") or 0)))
    return RAGAnswer(
        answer=parsed.get("answer", ""),
        citations=cits,
        confidence=parsed.get("confidence", "low"),
        answerable=bool(parsed.get("answerable", False)),
        model=settings.generator_model,
        tokens_in=r.usage_metadata.prompt_token_count or 0,
        tokens_out=r.usage_metadata.candidates_token_count or 0,
        latency_ms=0,
    )

In [ ]:
# In-scope vs out-of-scope test. Calls Gemini 3.6 Flash on Vertex — needs a real
# project; guarded so the notebook still runs without credentials.
grounded_chunks = [
    {'id': 'c1', 'source_uri': 'gs://docs/faq.pdf', 'page_start': 2,
     'text': 'DocuMind supports PDF, DOCX, TXT and Markdown files up to 50 MB each.'},
    {'id': 'c2', 'source_uri': 'gs://docs/faq.pdf', 'page_start': 3,
     'text': 'Scanned PDFs are OCR-processed automatically on upload.'},
]
try:
    import importlib, generator; importlib.reload(generator)
    ans = generator.generate('Which file types are supported?', grounded_chunks)
    print('IN-SCOPE  answerable=', ans.answerable, '| confidence=', ans.confidence,
          '| citations=', len(ans.citations))
    print('   ', ans.answer[:200])
    oos = generator.generate('What is the CEO home address?', grounded_chunks)
    print('OUT-SCOPE answerable=', oos.answerable, '| confidence=', oos.confidence,
          '| citations=', len(oos.citations))
    print('   ', oos.answer[:200])
except Exception as e:
    print('Live generate needs a Vertex project. Module is ready. Error:', type(e).__name__, e)

## Exercise 6: /v1/stream with Server-Sent Events

**Difficulty:** Medium

Build the SSE endpoint. Emit `event: citation` for each top-k chunk BEFORE generation, then `event: token` per streamed token, then `event: done` with the final `RAGAnswer`. Consume with `curl -N` and confirm ordering.

**Steps:**
1. Write the full `main.py` (`/health`, `/ready`, `/v1/query`, `/v1/stream`, IAP verify, OTel).
2. In `/v1/stream`, `retrieve` -> `rerank`, then yield one `citation` event per chunk.
3. Yield a `token` event per word of the answer, then a single `done` event with the serialized `RAGAnswer`.
4. Return a `StreamingResponse` with `media_type="text/event-stream"`; consume with `curl -N`.

**Expected:** `curl -N` streams the citation events first, then the token events, then one `done` event. `Content-Type: text/event-stream`, TTFB < 200ms.

In [ ]:
%%writefile main.py
import time, json, logging
from fastapi import FastAPI, Request, HTTPException, Depends
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from opentelemetry import trace
from opentelemetry.instrumentation.fastapi import FastAPIInstrumentor
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.cloud_trace import CloudTraceSpanExporter
from schemas import QueryRequest, RAGAnswer
from retriever import retrieve, rerank
from generator import generate
from config import settings

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")
log = logging.getLogger("documind-api")

trace.set_tracer_provider(TracerProvider())
trace.get_tracer_provider().add_span_processor(
    BatchSpanProcessor(CloudTraceSpanExporter(project_id=settings.project_id)))
tracer = trace.get_tracer(__name__)

app = FastAPI(title="DocuMind API", version="1.0.0")
FastAPIInstrumentor.instrument_app(app)

app.add_middleware(CORSMiddleware,
    allow_origins=["https://documind.example.com"],
    allow_methods=["POST","GET"], allow_headers=["*"])

def verify_iap(request: Request) -> dict:
    # In production, verify the signed X-Goog-IAP-JWT-Assertion header (see 12.4 auth.py).
    # For tests, allow X-User headers.
    email = request.headers.get("x-user-email")
    tenant = request.headers.get("x-tenant-id")
    if not email or not tenant:
        raise HTTPException(401, "missing identity headers")
    return {"email": email, "tenant_id": tenant}

@app.get("/health")
def health(): return {"status": "ok"}

@app.get("/ready")
def ready():
    # Lazy-init resource probes keep cold start fast; only warm when ready is probed
    from retriever import _genai_client, _index_endpoint, _fs
    _ = _genai_client(); _ = _index_endpoint(); _ = _fs()
    return {"status": "ready"}

@app.post("/v1/query", response_model=RAGAnswer)
def query(req: QueryRequest, user=Depends(verify_iap)):
    if req.tenant_id != user["tenant_id"]:
        raise HTTPException(403, "tenant mismatch")
    t0 = time.time()
    with tracer.start_as_current_span("retrieve"):
        chunks = retrieve(req.query, req.tenant_id, req.top_k, req.filters)
    with tracer.start_as_current_span("rerank"):
        chunks = rerank(req.query, chunks, req.top_k)
    with tracer.start_as_current_span("generate"):
        ans = generate(req.query, chunks)
    ans.latency_ms = int((time.time() - t0) * 1000)
    log.info(json.dumps({"event":"query","tenant":req.tenant_id,"user":user["email"],
                        "latency_ms":ans.latency_ms,"tokens_in":ans.tokens_in,
                        "tokens_out":ans.tokens_out,"answerable":ans.answerable,
                        "confidence":ans.confidence}))
    return ans

@app.post("/v1/stream")
def stream(req: QueryRequest, user=Depends(verify_iap)):
    if req.tenant_id != user["tenant_id"]:
        raise HTTPException(403, "tenant mismatch")
    def sse():
        chunks = retrieve(req.query, req.tenant_id, req.top_k, req.filters)
        chunks = rerank(req.query, chunks, req.top_k)
        for i, c in enumerate(chunks, 1):
            yield f"event: citation\ndata: {json.dumps({'n': i, 'source': c['source_uri'], 'page': c.get('page_start')})}\n\n"
        ans = generate(req.query, chunks)
        for tok in ans.answer.split():
            yield f"event: token\ndata: {json.dumps({'t': tok + ' '})}\n\n"
        yield f"event: done\ndata: {ans.model_dump_json()}\n\n"
    return StreamingResponse(sse(), media_type="text/event-stream")

In [ ]:
# Demonstrate the citation -> token -> done ordering the /v1/stream generator produces,
# using stubbed retrieve/generate so it runs without GCP. The SSE framing mirrors main.py.
import json
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from fastapi.testclient import TestClient
from schemas import RAGAnswer

stub_chunks = [
    {'source_uri': 'gs://docs/faq.pdf', 'page_start': 2},
    {'source_uri': 'gs://docs/faq.pdf', 'page_start': 3},
]
stub_answer = RAGAnswer(answer='PDF DOCX TXT and Markdown are supported', citations=[],
                        confidence='high', answerable=True, model='gemini-3.6-flash',
                        tokens_in=120, tokens_out=8, latency_ms=90)

demo = FastAPI()
@demo.post('/v1/stream')
def stream(request: Request):
    def sse():
        for i, c in enumerate(stub_chunks, 1):
            yield f"event: citation\ndata: {json.dumps({'n': i, 'source': c['source_uri'], 'page': c['page_start']})}\n\n"
        for tok in stub_answer.answer.split():
            yield f"event: token\ndata: {json.dumps({'t': tok + ' '})}\n\n"
        yield f"event: done\ndata: {stub_answer.model_dump_json()}\n\n"
    return StreamingResponse(sse(), media_type='text/event-stream')

resp = TestClient(demo).post('/v1/stream')
print('Content-Type:', resp.headers['content-type'])
for line in resp.text.splitlines():
    if line.startswith('event:'):
        print(line)

## Exercise 7: OpenTelemetry spans + Cloud Trace

**Difficulty:** Challenge

Wire `opentelemetry-instrumentation-fastapi` + `opentelemetry-exporter-gcp-trace`. Bracket retrieve / rerank / generate with custom spans. Deploy, send 10 queries, open Cloud Trace, and confirm the waterfall shows three child spans under the HTTP parent.

**Steps:**
1. Set a `TracerProvider` with a `BatchSpanProcessor` -> `CloudTraceSpanExporter` (all wired in `main.py`, Ex 6).
2. `FastAPIInstrumentor.instrument_app(app)` creates the HTTP parent span automatically.
3. Wrap each stage in `tracer.start_as_current_span("retrieve"|"rerank"|"generate")`.
4. Deploy, send 10 queries, and inspect the trace waterfall in Cloud Trace.

**Expected:** Cloud Trace shows `POST /v1/query` with `retrieve` / `rerank` / `generate` children. Span attributes include `tenant_id` and `top_k`. Dev sampling set to 1.0.

In [ ]:
# The tracing wiring lives in main.py (Ex 6). Confirm the OTel pieces import and that
# the three custom spans are present in the source. Recording spans is verified in
# Cloud Trace after deploy (bash cell below), since the exporter needs a real project.
import inspect, main
from opentelemetry import trace
from opentelemetry.exporter.cloud_trace import CloudTraceSpanExporter  # noqa: F401

src = inspect.getsource(main.query)
for stage in ('retrieve', 'rerank', 'generate'):
    present = f'start_as_current_span("{stage}")' in src
    print(f'custom span for {stage:9}:', 'yes' if present else 'MISSING')
print('tracer provider:', type(trace.get_tracer_provider()).__name__)

In [ ]:
%%bash
# After deploy, send 10 queries then open the trace waterfall for POST /v1/query.
# (Replace the URL/token with your deployed service; see Ex 8 for the token.)
PROJECT=documind-ai-YOUR-ID
for i in $(seq 1 10); do
  curl -sSf -H "Authorization: Bearer $TOK" \
    -H "X-User-Email: alice@acme.in" -H "X-Tenant-Id: tenant-acme" \
    -H "Content-Type: application/json" \
    -d '{"query":"Which file types are supported?","tenant_id":"tenant-acme","user_id":"u_1","top_k":5,"stream":false}' \
    https://documind-api-xxx.run.app/v1/query > /dev/null
done
# List recent traces (each should show retrieve / rerank / generate children):
gcloud trace list --project=$PROJECT --limit=10

## Exercise 8: Cloud Run deploy + IAM + tenant isolation test

**Difficulty:** Challenge

Dockerise with gunicorn + `UvicornWorker`. Deploy with `--no-allow-unauthenticated` + `documind-api-sa` + VPC connector. Grant `ui-sa` `roles/run.invoker`. Write a test that sends `tenant_id=B` with a JWT claim for tenant-A and expect a 403.

**Steps:**
1. Write the `Dockerfile` (tini + gunicorn `UvicornWorker`, non-root user, HEALTHCHECK).
2. `gcloud run deploy` with `--no-allow-unauthenticated`, `documind-api-sa`, VPC connector, private egress.
3. Grant `documind-ui-sa` `roles/run.invoker`; call with an impersonated identity token.
4. Send a request where `tenant_id` disagrees with the verified claim — expect 403.

**Expected:** No identity token -> 401. `ui-sa` token + matching tenant -> success. Mismatched tenant vs claim -> 403 "tenant mismatch".

In [ ]:
%%writefile Dockerfile
FROM python:3.12-slim
RUN apt-get update && apt-get install -y --no-install-recommends tini ca-certificates && rm -rf /var/lib/apt/lists/*
RUN useradd --create-home --shell /bin/bash --uid 10001 app
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY --chown=app:app . .
USER app
ENV PORT=8080 PYTHONUNBUFFERED=1
EXPOSE 8080
HEALTHCHECK --interval=20s --timeout=3s --start-period=15s CMD python -c "import urllib.request as r; r.urlopen('http://localhost:8080/health').read()" || exit 1
ENTRYPOINT ["/usr/bin/tini", "--"]
CMD ["gunicorn", "-k", "uvicorn.workers.UvicornWorker", "-w", "2", "-b", "0.0.0.0:8080", "-t", "120", "--access-logfile", "-", "main:app"]

In [ ]:
%%bash
# Build, deploy private, and grant the UI service account invoke rights.
PROJECT=documind-ai-YOUR-ID
GIT_SHA=$(git rev-parse --short HEAD 2>/dev/null || echo dev)
EID=0000000000000000000

gcloud run deploy documind-api \
  --image=us-central1-docker.pkg.dev/$PROJECT/documind/api:$GIT_SHA \
  --region=us-central1 --platform=managed \
  --no-allow-unauthenticated \
  --ingress=internal-and-cloud-load-balancing \
  --memory=1Gi --cpu=2 --concurrency=40 --timeout=120 \
  --min-instances=0 --max-instances=20 \
  --cpu-boost --execution-environment=gen2 \
  --service-account=documind-api-sa@$PROJECT.iam.gserviceaccount.com \
  --set-env-vars="VECTOR_INDEX_ENDPOINT=projects/$PROJECT/locations/us-central1/indexEndpoints/$EID,VECTOR_DEPLOYED_INDEX_ID=dep_001" \
  --vpc-connector=projects/$PROJECT/locations/us-central1/connectors/documind-vpc \
  --vpc-egress=private-ranges-only

# Let the Streamlit UI service account call the API
gcloud run services add-iam-policy-binding documind-api \
  --region=us-central1 \
  --member="serviceAccount:documind-ui-sa@$PROJECT.iam.gserviceaccount.com" \
  --role="roles/run.invoker"

In [ ]:
%%bash
# Smoke tests against the deployed, authenticated service.
PROJECT=documind-ai-YOUR-ID
# Identity token (not a user token) because of --no-allow-unauthenticated
TOK=$(gcloud auth print-identity-token --impersonate-service-account=documind-ui-sa@$PROJECT.iam.gserviceaccount.com)

# No token -> 401
curl -s -o /dev/null -w '%{http_code}\n' https://documind-api-xxx.run.app/health

# Valid token + matching tenant -> 200
curl -sSf -H "Authorization: Bearer $TOK" \
     -H "X-User-Email: alice@acme.in" -H "X-Tenant-Id: tenant-acme" \
     -H "Content-Type: application/json" \
     -d '{"query":"Which file types are supported?","tenant_id":"tenant-acme","user_id":"u_1","top_k":5,"stream":false}' \
     https://documind-api-xxx.run.app/v1/query

# Mismatched tenant vs claim -> 403 "tenant mismatch"
curl -s -o /dev/null -w '%{http_code}\n' -H "Authorization: Bearer $TOK" \
     -H "X-User-Email: alice@acme.in" -H "X-Tenant-Id: tenant-acme" \
     -H "Content-Type: application/json" \
     -d '{"query":"secret","tenant_id":"tenant-other","user_id":"u_1","top_k":5,"stream":false}' \
     https://documind-api-xxx.run.app/v1/query

In [ ]:
# Reproduce the 401 / 403 boundary locally (no GCP) using the exact verify_iap +
# tenant-match logic from main.py, so the test is runnable in Colab.
from fastapi import FastAPI, Request, HTTPException, Depends
from fastapi.testclient import TestClient
from schemas import QueryRequest

def verify_iap(request: Request) -> dict:
    email = request.headers.get('x-user-email')
    tenant = request.headers.get('x-tenant-id')
    if not email or not tenant:
        raise HTTPException(401, 'missing identity headers')
    return {'email': email, 'tenant_id': tenant}

iso = FastAPI()
@iso.post('/v1/query')
def query(req: QueryRequest, user=Depends(verify_iap)):
    if req.tenant_id != user['tenant_id']:
        raise HTTPException(403, 'tenant mismatch')
    return {'ok': True, 'tenant': req.tenant_id}

c = TestClient(iso)
body = {'query': 'hi', 'tenant_id': 'tenant-acme', 'user_id': 'u_1', 'top_k': 5}

no_id = c.post('/v1/query', json=body)  # no identity headers
match = c.post('/v1/query', json=body, headers={'X-User-Email': 'a@acme.in', 'X-Tenant-Id': 'tenant-acme'})
mismatch = c.post('/v1/query', json={**body, 'tenant_id': 'tenant-other'},
                  headers={'X-User-Email': 'a@acme.in', 'X-Tenant-Id': 'tenant-acme'})

print('no identity   :', no_id.status_code, no_id.json())        # 401
print('matching tenant:', match.status_code, match.json())        # 200
print('tenant claim   :', mismatch.status_code, mismatch.json())  # 403 tenant mismatch